In [2]:
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X_bee = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature_bee\mfcc_clbp_1D_bee_80_nodelta.npy")
X_nobee = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature_bee\mfcc_clbp_1D_nobee_80_nodelta.npy")
X_noqueen = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature_bee\mfcc_clbp_1D_noqueen_80_nodelta.npy")

X = np.vstack([X_bee, X_nobee, X_noqueen]).astype(np.float32)
y = np.hstack([
    np.zeros(len(X_bee), dtype=np.int8),
    np.ones(len(X_nobee), dtype=np.int8),
    np.full(len(X_noqueen), 2, dtype=np.int8)
])

X, y = shuffle(X, y, random_state=42)

print("Final shape:", X.shape, y.shape)

Final shape: (13792, 514) (13792,)


In [3]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.preprocessing import label_binarize


# ===== Hàm tính EER multi-class =====
def compute_eer_multiclass(y_true, y_score, n_classes):
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    eers = []

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        fnr = 1 - tpr
        eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
        eers.append(eer)

    return np.mean(eers)


# ===== Hold-out 80/20 =====
n_classes = 3

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # giữ tỉ lệ lớp
    random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        n_jobs=-1,
        random_state=42
    ))
])

print("Training...")

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro")
rec  = recall_score(y_test, y_pred, average="macro")
f1   = f1_score(y_test, y_pred, average="macro")
auc  = roc_auc_score(y_test, y_score, multi_class="ovr")
eer  = compute_eer_multiclass(y_test, y_score, n_classes)


print("===== Hold-out 80/20 Result =====")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)
print("ROC-AUC  :", auc)
print("EER      :", eer)

Training...
===== Hold-out 80/20 Result =====
Accuracy : 0.7017035157665821
Precision: 0.65723259593796
Recall   : 0.6530364498458215
F1-score : 0.637847353620442
ROC-AUC  : 0.8370075574074765
EER      : 0.2347848979874629


In [4]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Multi-class EER =====
def compute_eer_multiclass(y_true, y_score, n_classes):
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    eers = []

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        fnr = 1 - tpr
        eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
        eers.append(eer)

    return np.mean(eers)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1
n_classes = 3

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred, average="macro"))
    recalls.append(recall_score(y_test, y_pred, average="macro"))
    f1s.append(f1_score(y_test, y_pred, average="macro"))
    aucs.append(roc_auc_score(y_test, y_score, multi_class="ovr"))
    eers.append(compute_eer_multiclass(y_test, y_score, n_classes))


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.7962585365686982
Precision: 0.7759028703323816
Recall   : 0.7474348373523003
F1-score : 0.7369051656905914
ROC-AUC  : 0.8926992945682297
EER      : 0.17013494077586955


In [9]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Multi-class EER =====
def compute_eer_multiclass(y_true, y_score, n_classes):
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    eers = []

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        fnr = 1 - tpr
        eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
        eers.append(eer)

    return np.mean(eers)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1
n_classes = 3

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            probability=True,   # cần cho ROC + EER
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred, average="macro"))
    recalls.append(recall_score(y_test, y_pred, average="macro"))
    f1s.append(f1_score(y_test, y_pred, average="macro"))
    aucs.append(roc_auc_score(y_test, y_score, multi_class="ovr"))
    eers.append(compute_eer_multiclass(y_test, y_score, n_classes))


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.6485643793231513
Precision: 0.611129923368301
Recall   : 0.612256446951275
F1-score : 0.606355473317582
ROC-AUC  : 0.8024090635505461
EER      : 0.2664128734848648


In [10]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Multi-class EER =====
def compute_eer_multiclass(y_true, y_score, n_classes):
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    eers = []

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        fnr = 1 - tpr
        eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
        eers.append(eer)

    return np.mean(eers)


# ===== DNN model =====
def create_dnn(input_dim, n_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),

        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, precisions, recalls, f1s, aucs, eers = [], [], [], [], [], []
n_classes = 3
count = 1

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # model
    model = create_dnn(X.shape[1], n_classes)

    model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        verbose=0
    )

    y_score = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_score, axis=1)

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred, average="macro"))
    recalls.append(recall_score(y_test, y_pred, average="macro"))
    f1s.append(f1_score(y_test, y_pred, average="macro"))
    aucs.append(roc_auc_score(y_test, y_score, multi_class="ovr"))
    eers.append(compute_eer_multiclass(y_test, y_score, n_classes))


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.6413867096174929
Precision: 0.6044239119913638
Recall   : 0.6058667501879309
F1-score : 0.5993344233529685
ROC-AUC  : 0.7900967735787342
EER      : 0.277441308006798
